In [16]:
import numpy as np
import math
import trimesh
import plotly.graph_objects as go
import Grasping_Face.grasping as gf
import Grasping_Face.visualize_grasping as gfv
import json

In [17]:
name = 'logitech_c930e_m' # logitech_c930e_m / obj_000001

target_mesh_file = f'./models_target/models_cad/{name}.obj'

In [ ]:
result = gf.compute_best_patch_pairs(
    mesh_path=target_mesh_file,
    mesh_max_triangles = 1000,
    angle_deg=30,              # 패치 병합 허용 각도 (↑면 패치 수 ↓)
    coplanar_tol=0.1,            # 공면성 허용 오차 (↑면 패치 수 ↓)
    min_opening=5.0,          # 그리퍼 최소 개구(mm)
    max_opening=140.0,         # 그리퍼 최대 개구(mm)
    angle_tolerance_deg=10,    # 패치 페어 정반대 threshold
    top_k=100                  # 상위 패치 페어 후보 개수
)

# reports = gf.check_gripper_feasibility_with_yaw(result)
reports = gf.check_gripper_feasibility_faces_with_yaw(result)

In [42]:
fig = gfv.visualize_merged_patches_plotly(result, show=True)
# fig.write_html(f"./Grasping_Result/{name}_patch_normal.html", include_plotlyjs="cdn", full_html=True)

In [33]:
fig = gfv.visualize_pairs_centroid_lines(result, show=True)
# fig.write_html(f"./Grasping_Result/{name}_patch_pairs.html", include_plotlyjs="cdn", full_html=True)

In [34]:
fig = gfv.visualize_feasible_pairs_pads(result, reports, show=True)
# fig.write_html(f"./Grasping_Result/{name}_patch_pairs_feasible.html", include_plotlyjs="cdn", full_html=True)

In [5]:
fig = gfv.visualize_feasible_pairs_pads_gripper(result, reports, show=True)
# fig.write_html(f"./Grasping_Result/{name}_patch_pairs_grip_feasible.html", include_plotlyjs="cdn", full_html=True)

In [6]:
reports

[{'pair_index': 0,
  'patch_i': 109,
  'patch_j': 113,
  'face_i': np.int64(541),
  'face_j': np.int64(578),
  'feasible': True,
  'feasible_yaw': 0,
  'moment': 0.3789788247432559},
 {'pair_index': 11,
  'patch_i': 5,
  'patch_j': 83,
  'face_i': 30,
  'face_j': np.int64(395),
  'feasible': True,
  'feasible_yaw': 90,
  'moment': 0.40818621907799135},
 {'pair_index': 38,
  'patch_i': 54,
  'patch_j': 55,
  'face_i': np.int64(196),
  'face_j': np.int64(240),
  'feasible': True,
  'feasible_yaw': 0,
  'moment': 0.576266376742149},
 {'pair_index': 37,
  'patch_i': 51,
  'patch_j': 55,
  'face_i': np.int64(206),
  'face_j': np.int64(256),
  'feasible': True,
  'feasible_yaw': 0,
  'moment': 0.5762935334119533},
 {'pair_index': 2,
  'patch_i': 102,
  'patch_j': 106,
  'face_i': np.int64(495),
  'face_j': np.int64(508),
  'feasible': True,
  'feasible_yaw': 0,
  'moment': 0.7511885306462849},
 {'pair_index': 12,
  'patch_i': 49,
  'patch_j': 83,
  'face_i': 161,
  'face_j': np.int64(387),
 

In [6]:
pairind = 2
p_i = result['patches'][reports[pairind]['patch_i']]
p_j = result['patches'][reports[pairind]['patch_j']]
yaw_deg = reports[pairind]['feasible_yaw']

H_OG, stroke = gf.build_gripper_pose_obj(p_i, p_j, yaw_deg)
H_OG[:3, :3], H_OG[:3, 3] # O좌표계 기준 G좌표계 원점 위치

(array([[ 4.14197686e-04, -6.29992931e-06, -9.99999914e-01],
        [-1.52081987e-02, -9.99884349e-01, -6.12340114e-17],
        [-9.99884263e-01,  1.52081974e-02, -4.14245594e-04]]),
 array([77.43948809,  9.68957949,  8.11205237]))

In [7]:
# SAM6D result 
with open("./RUN_result/output/detection_pem_250910_0_logitech_c930e_m.json", "r") as f:
    detections = json.load(f)

# score 가장 높은 detection 선택
best_det = max(detections, key=lambda d: d["score"])
R_oc = np.array(best_det["R"])  # (3x3)
t_oc = np.array(best_det["t"]).reshape(3, 1)  # (3x1)

H_OC = gf.to44(R_oc, t_oc) # C좌표계 기준 O좌표계 원점 위치
H_OC[:3, :3], H_OC[:3, 3]

(array([[-0.83090752,  0.51341951, -0.21445768],
        [-0.37969002, -0.24145162,  0.8930493 ],
        [ 0.40672785,  0.82346916,  0.39556435]]),
 array([-189.87509155,  -56.61289978,  599.92248535]))

In [8]:
# 좌표계 시각화
H_E = np.eye(4,4)

# 고정변환
H_GnEn = gf.to44(gf.Rotx(180) @ gf.Rotz(90), [0,0,-135])   # EE -> Grip  TODO: 로봇 컨트롤러 신호 받아 변환행렬 만들기
H_GnCn = gf.to44(np.eye(3), [0,48,6])           # Cam -> Grip
H_CnGn = np.linalg.inv(H_GnCn)
H_CnEn = H_GnEn @ H_CnGn                        # EE -> Cam
H_EG = np.linalg.inv(H_GnEn)

H_OdCn = H_OC
H_OdEn = H_CnEn @ H_OdCn
# H_GdOd = np.linalg.inv(H_OG)
H_GdEn = H_OdEn @ H_OG
H_EdEn = H_GdEn @ H_EG

H_dict = {
    "E": H_E, # 엔드이펙터
    "G": H_GnEn, 
    "C": H_CnEn,
    "O_d": H_OdEn, # 목표 
    "G_d": H_GdEn,
    "E_d": H_EdEn, # EE 상대 Pose
}
fig = gfv.visualize_frames(H_dict, scale=100)

In [9]:
H_EdEn[:3, :3], H_EdEn[:3, 3]

(array([[ 0.25500776, -0.88943116,  0.37932005],
        [-0.51661641,  0.20628051,  0.83099629],
        [-0.81736067, -0.40787358, -0.40689168]]),
 array([ 180.31920899,  363.1695168 , -826.53756679]))

In [5]:
import os
import numpy as np
import math
import trimesh
import plotly.graph_objects as go
import Grasping_Face.grasping as gf
import Grasping_Face.visualize_grasping as gfv

cads = [file for file in os.listdir('./models_target/models_cad/') if file.endswith('.obj')]

for cad in cads:
    name = cad.split('.')[0]
    target_mesh_file = f'./models_target/models_cad/{name}.obj'


    result = gf.compute_best_patch_pairs(
        mesh_path=target_mesh_file,
        mesh_max_triangles = 1000,
        angle_deg=30,             # 패치 병합 허용 각도 (↑면 패치 수 ↓)
        coplanar_tol=1,           # 공면성 허용 오차 (↑면 패치 수 ↓)
        min_opening=5.0,          # 그리퍼 최소 개구(mm)
        max_opening=140.0,        # 그리퍼 최대 개구(mm)
        angle_tolerance_deg=10,   # 패치 페어 정반대 threshold
        top_k=100                 # 상위 패치 페어 후보 개수
    )

    # reports = gf.check_gripper_feasibility(result)
    # reports = gf.check_gripper_feasibility_with_yaw(result)
    reports = gf.check_gripper_feasibility_faces_with_yaw(result)

    feasible_p = result['top_k']
    feasible_r = [r for r in reports if r.get('feasible')]
    print(f'{name} : patch pairs {len(feasible_p)}, feasible sol. {len(feasible_r)}')

    fig = gfv.visualize_merged_patches_plotly(result)
    fig.write_html(f"./Grasping_Result/3-patch_normal_{name}.html", include_plotlyjs="cdn", full_html=True)

    fig = gfv.visualize_pairs_centroid_lines(result)
    fig.write_html(f"./Grasping_Result/2-patch_pairs_{name}.html", include_plotlyjs="cdn", full_html=True)

    try:
        fig = gfv.visualize_feasible_pairs_pads(result, reports)
        fig.write_html(f"./Grasping_Result/1-patch_pairs_feasible_{name}.html", include_plotlyjs="cdn", full_html=True)
    except:
        pass

logitech_c930e_m : patch pairs 46, feasible sol. 15
obj_000001 : patch pairs 73, feasible sol. 7
obj_000002 : patch pairs 100, feasible sol. 17
obj_000003 : patch pairs 89, feasible sol. 8
obj_000004 : patch pairs 40, feasible sol. 9
obj_000005 : patch pairs 79, feasible sol. 9
obj_000006 : patch pairs 100, feasible sol. 11
obj_000007 : patch pairs 100, feasible sol. 3
obj_000008 : patch pairs 100, feasible sol. 6
obj_000009 : patch pairs 100, feasible sol. 6
obj_000010 : patch pairs 100, feasible sol. 5
obj_000011 : patch pairs 85, feasible sol. 4
obj_000012 : patch pairs 100, feasible sol. 7
obj_000013 : patch pairs 69, feasible sol. 19
obj_000014 : patch pairs 93, feasible sol. 5
obj_000015 : patch pairs 74, feasible sol. 16
obj_000016 : patch pairs 100, feasible sol. 24
obj_000017 : patch pairs 100, feasible sol. 3
obj_000018 : patch pairs 100, feasible sol. 0
obj_000019 : patch pairs 53, feasible sol. 19
obj_000020 : patch pairs 54, feasible sol. 19
obj_000021 : patch pairs 71, fe

d:\PythonDev\OPE_Grasping\Grasping_Face\grasping.py:480: RuntimeWarning:

invalid value encountered in divide



obj_000029 : patch pairs 51, feasible sol. 9
obj_000030 : patch pairs 45, feasible sol. 33


#### Test

In [20]:
name = 'logitech_c930e_m' # logitech_c930e_m / obj_000001
target_mesh_file = f'./models_target/models_cad/{name}.obj'

In [21]:
mesh_path = target_mesh_file
mesh_max_triangles = 1000
angle_deg=30              # 패치 병합 허용 각도 (↑면 패치 수 ↓)
coplanar_tol=1            # 공면성 허용 오차 (↑면 패치 수 ↓)
min_opening=5.0          # 그리퍼 최소 개구(mm)
max_opening=140.0         # 그리퍼 최대 개구(mm)
angle_tolerance_deg=10    # 패치 페어 정반대 threshold
top_k=100                 # 상위 패치 페어 후보 개수

In [22]:
mesh_og = trimesh.load(target_mesh_file)

In [23]:
remesh, mesh_quad = gf.load_uniform_mesh_with_open3d(mesh_path, target_triangles=mesh_max_triangles)

In [24]:
remesh2 = gf.split_long_edges(remesh)

In [30]:
fig = gfv.visualize_mesh_with_edges(remesh2, edge_width=3)

In [12]:
patches = gf.extract_planar_patches(remesh2, angle_deg=angle_deg, coplanar_tol=coplanar_tol)
patches = gf.orient_patch_normals(mesh_quad, patches, inward=False)  # false: 모두 바깥쪽으로 정렬

len(patches)

85

In [ ]:
params = gf.PatchPairParams(
    min_opening=min_opening,
    max_opening=max_opening,
    angle_tolerance_deg=angle_tolerance_deg,
)

In [ ]:
np.linalg.norm(patches[70].centroid)

In [ ]:
patches[70].centroid

In [ ]:
cand = gf.score_patch_pair(patches[70], patches[74], params, remesh)
cand

In [ ]:
cands = []
nvecs = [gf.unit(np.array(p.normal)) for p in patches] # unit normals 
# 현재 패치와 normals가 angle_tolerance_deg 이하인 patch만 남김 (각도 180도 +- angle_tolerance_deg 범위)
for i in range(len(patches) - 1):
    ni = nvecs[i]

    best_cand = None
    best_score = -1.0

    for j in range(i + 1, len(patches)):
        # i-j 법선 각도
        dot = float(ni @ nvecs[j])
        ang = math.degrees(math.acos(max(-1.0, min(1.0, dot))))
        # 180° - tol 보다 작으면 충분히 반대가 아님 -> 스킵
        if ang < 180.0 - angle_tolerance_deg:
            continue

        cand = gf.score_patch_pair(patches[i], patches[j], params, remesh)

        # 기존 폭/면적/점수 필터
        if cand.width < params.min_opening or cand.width > params.max_opening:
            continue
        if cand.score <= 0.0:
            continue

        print(i,j)


        # i에 대해 최고 점수만 유지
        if cand.score > best_score:
            best_score = cand.score
            best_cand = cand

    if best_cand is not None:
        cands.append(best_cand)

cands.sort(key=lambda c: c.score, reverse=True)

In [ ]:
len(cands)